<a href="https://colab.research.google.com/github/maclandrol/cours-ia-med/blob/master/05_Analyse_Radiographies_Thoraciques.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 05. Analyse de Radiographies Thoraciques

**Enseignant:** Emmanuel Noutahi, PhD

---

## Objectifs

- Utiliser TorchXRayVision pour détecter 18 pathologies thoraciques
- Charger et analyser des radiographies
- Comprendre les limites: qualité d'image et pathologies hors distribution

## Pathologies Détectables (18)

Atélectasie, Cardiomégalie, Épanchement, Infiltration, Masse, Nodule, Pneumonie, Pneumothorax, Consolidation, Œdème, Emphysème, Fibrose, Épaississement pleural, Hernie, Opacité pulmonaire, Lésion pleurale, Fracture, Dispositif médical

## 1. Installation

In [ ]:
!pip install -q torchxrayvision torch torchvision scikit-image pillow matplotlib

In [ ]:
import torchxrayvision as xrv
import torch
import torchvision
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import skimage.io
import skimage.transform
import skimage.util
import warnings
warnings.filterwarnings('ignore')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

## 2. Chargement du Modèle

In [ ]:
model = xrv.models.DenseNet(weights="densenet121-res224-all")
model = model.to(device)
model.eval()

print(f"Pathologies détectables ({len(model.pathologies)}):")
for i, p in enumerate(model.pathologies, 1):
    print(f"{i:2d}. {p}")

## 3. Chargement d'Images

### Option A: Depuis un Dataset TorchXRayVision (Recommandé)

In [ ]:
# Utiliser un petit dataset pour démonstration
# NIH_Google contient un échantillon d'images

# Définir les transformations
transform = torchvision.transforms.Compose([
    xrv.datasets.XRayCenterCrop(),
    xrv.datasets.XRayResizer(224)
])

# Charger le dataset
# Note: Si vous n'avez pas les données, passez à l'Option B (Upload)
try:
    dataset = xrv.datasets.NIH_Google_Dataset(
        imgpath="./images",
        csvpath="./nih_google.csv",
        transform=transform
    )
    
    # Charger une image
    sample = dataset[0]
    img_tensor = sample['img'].to(device)
    
    # Afficher
    plt.figure(figsize=(6, 6))
    plt.imshow(img_tensor[0].cpu(), cmap='gray')
    plt.title("Radiographie du Dataset")
    plt.axis('off')
    plt.show()
    
    print("Dataset chargé avec succès!")
    using_dataset = True
    
except:
    print("Dataset non disponible. Utilisez l'Option B (Upload) ci-dessous.")
    using_dataset = False

### Option B: Upload de Votre Image

In [ ]:
if not using_dataset:
    from google.colab import files
    
    uploaded = files.upload()
    
    if uploaded:
        img_path = list(uploaded.keys())[0]
        
        # Charger et préparer
        img = skimage.io.imread(img_path)
        img = xrv.datasets.normalize(img, 255)
        
        if len(img.shape) == 3:
            img = img.mean(2)
        img = img[None, ...]
        
        img = transform(img)
        img_tensor = torch.from_numpy(img).to(device)
        
        # Afficher
        plt.figure(figsize=(6, 6))
        plt.imshow(img[0], cmap='gray')
        plt.title("Votre Image")
        plt.axis('off')
        plt.show()
        
        print(f"Image chargée: {img_path}")

## 4. Analyse

In [ ]:
# Prédiction
with torch.no_grad():
    pred = model(img_tensor[None,...])
    probs = torch.sigmoid(pred).cpu().numpy()[0]

# Trier par probabilité
indices = np.argsort(probs)[::-1]

print("RÉSULTATS")
print("="*55)
for idx in indices[:10]:
    pathology = model.pathologies[idx]
    prob = probs[idx]
    bar = '█' * int(prob * 30)
    print(f"{pathology:<25} {prob:>6.1%} {bar}")
print("="*55)

## 5. Visualisation

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Image
ax1.imshow(img_tensor[0].cpu(), cmap='gray')
ax1.set_title("Radiographie")
ax1.axis('off')

# Top 8
top_n = 8
top_indices = indices[:top_n]
top_pathologies = [model.pathologies[i] for i in top_indices]
top_probs = [probs[i] for i in top_indices]

colors = ['red' if p > 0.5 else 'orange' if p > 0.3 else 'steelblue' for p in top_probs]
ax2.barh(range(top_n), top_probs, color=colors)
ax2.set_yticks(range(top_n))
ax2.set_yticklabels(top_pathologies)
ax2.set_xlabel('Probabilité')
ax2.set_title('Top 8 Pathologies')
ax2.set_xlim(0, 1)
ax2.axvline(x=0.5, color='red', linestyle='--', alpha=0.3, label='Seuil 50%')
ax2.axvline(x=0.3, color='orange', linestyle='--', alpha=0.3, label='Seuil 30%')
ax2.legend(loc='lower right')
ax2.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Impact des Distorsions d'Image

Comment la qualité d'image affecte-t-elle les prédictions?

In [ ]:
# Convertir tensor en array pour manipulation
img_array = (img_tensor[0].cpu().numpy() * 255).astype(np.uint8)

# Créer distorsions
distortions = {}
distortions['Original'] = img_array

# Rotation
distortions['Rotation 15°'] = skimage.transform.rotate(
    img_array, 15, mode='edge', preserve_range=True
).astype(np.uint8)

# Zoom (crop + resize)
h, w = img_array.shape
crop_size = int(min(h, w) * 0.7)
start = (h - crop_size) // 2
cropped = img_array[start:start+crop_size, start:start+crop_size]
distortions['Zoom'] = skimage.transform.resize(
    cropped, img_array.shape, preserve_range=True
).astype(np.uint8)

# Bruit
noisy = skimage.util.random_noise(img_array / 255.0, mode='gaussian', var=0.01)
distortions['Bruit'] = (noisy * 255).astype(np.uint8)

# Faible contraste
mean_val = img_array.mean()
low_contrast = ((img_array - mean_val) * 0.5 + mean_val).clip(0, 255)
distortions['Faible Contraste'] = low_contrast.astype(np.uint8)

print(f"Créé {len(distortions)} versions de l'image")

In [ ]:
# Analyser chaque distorsion
results = {}

for name, img_dist in distortions.items():
    # Préparer pour le modèle
    img_norm = xrv.datasets.normalize(img_dist, 255)
    img_norm = img_norm[None, ...]
    img_tensor_dist = torch.from_numpy(img_norm).to(device)
    
    # Prédiction
    with torch.no_grad():
        pred = model(img_tensor_dist[None,...])
        probs_dist = torch.sigmoid(pred).cpu().numpy()[0]
    
    # Top 3
    top_idx = np.argsort(probs_dist)[-3:][::-1]
    top_preds = [(model.pathologies[i], probs_dist[i]) for i in top_idx]
    
    results[name] = {'image': img_dist, 'predictions': top_preds}

print("Analyse terminée!")

In [ ]:
# Visualisation
fig, axes = plt.subplots(2, 2, figsize=(12, 12))
axes = axes.flatten()

for idx, (name, data) in enumerate(results.items()):
    if idx >= 4:
        break
    
    axes[idx].imshow(data['image'], cmap='gray')
    
    top_pred = data['predictions'][0]
    title = f"{name}\n{top_pred[0]}: {top_pred[1]*100:.1f}%"
    axes[idx].set_title(title, fontsize=11, fontweight='bold')
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

# Tableau comparatif
print("\nIMPACT DES DISTORSIONS")
print("="*70)
for name, data in results.items():
    print(f"\n{name:20s}")
    for i, (patho, prob) in enumerate(data['predictions'], 1):
        print(f"  {i}. {patho:25s} {prob*100:5.1f}%")
print("\n" + "="*70)

## 7. Pathologies Out-of-Distribution (OOD)

Que se passe-t-il avec une pathologie que le modèle n'a pas vue pendant l'entraînement?

### Exemple: COVID-19

Le modèle a été entraîné AVANT la pandémie COVID. Testons avec une vraie radiographie COVID.

**Dataset COVID-19**: https://github.com/ieee8023/covid-chestxray-dataset

In [ ]:
# Télécharger une radiographie COVID réelle
# Source: https://github.com/ieee8023/covid-chestxray-dataset

!wget -q https://raw.githubusercontent.com/ieee8023/covid-chestxray-dataset/master/images/01E392EE-69F9-4E33-BFCE-E5C968654078.jpeg -O covid_xray.jpg

# Charger l'image
covid_img_raw = Image.open('covid_xray.jpg').convert('L')

plt.figure(figsize=(6, 6))
plt.imshow(covid_img_raw, cmap='gray')
plt.title("Radiographie COVID-19 (Dataset Réel)")
plt.axis('off')
plt.show()

print("Image COVID-19 téléchargée.")
print("Le modèle n'a JAMAIS vu de COVID pendant l'entraînement.")
print("Voyons comment il réagit...")

In [ ]:
# Préparer l'image COVID pour le modèle
covid_img = np.array(covid_img_raw)
covid_img = xrv.datasets.normalize(covid_img, 255)

if len(covid_img.shape) == 3:
    covid_img = covid_img.mean(2)
covid_img = covid_img[None, ...]

covid_img = transform(covid_img)
covid_tensor = torch.from_numpy(covid_img).to(device)

# Analyser
with torch.no_grad():
    pred_covid = model(covid_tensor[None,...])
    probs_covid = torch.sigmoid(pred_covid).cpu().numpy()[0]

# Top prédictions
indices_covid = np.argsort(probs_covid)[::-1]

print("\nPRÉDICTIONS POUR RADIOGRAPHIE COVID-19")
print("="*55)
print("Rappel: Le modèle N'A PAS été entraîné sur COVID!")
print("="*55)
for idx in indices_covid[:8]:
    pathology = model.pathologies[idx]
    prob = probs_covid[idx]
    bar = '█' * int(prob * 30)
    print(f"{pathology:<25} {prob:>6.1%} {bar}")
print("="*55)

print("\nOBSERVATION:")
print("Le modèle mappe COVID vers des catégories connues:")
print("  - Pneumonia, Infiltration, Edema, Lung Opacity")
print("\nC'est typique d'une pathologie OUT-OF-DISTRIBUTION (OOD).")
print("Le modèle utilise les catégories les plus proches qu'il connaît.")

## 8. Points Clés

### Ce que vous avez appris:
- Charger et analyser des radiographies avec TorchXRayVision
- Interpréter les probabilités de détection
- Impact de la qualité d'image sur les prédictions
- Comportement avec pathologies OOD (COVID exemple)

### Applications:
- **Triage**: Prioriser cas urgents
- **Aide au diagnostic**: Détection rapide
- **Recherche**: Analyser grandes cohortes

### Limites importantes:
- **Qualité d'image**: Rotation, bruit, contraste affectent les résultats
- **Pathologies OOD**: Le modèle mappe vers catégories connues
- **Populations**: Entraîné sur certaines populations
- **Outil d'aide**: Ne remplace pas l'expertise médicale

### Observations clés:
1. **Rotation légère**: Impact minimal
2. **Bruit/Contraste**: Peuvent changer prédictions significativement
3. **COVID (OOD)**: Détecté comme Pneumonia/Infiltration/Edema
4. **Validation**: Toujours vérifier contexte clinique